In [16]:
import torch
torch.set_default_dtype(torch.float64)
from cpu_cb import E8P12_codebook
from itertools import product

In [17]:
def hb_transform(x):
    if len(x.shape) == 1:
        x = x.view(1,-1)
    (m,n) = x.shape
    k = 1
    while 4**k < n:
        k += 1
    assert(4**k == n)
    b = torch.tensor([1,1,-1,-1]).to(x.dtype)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1) + (4,) + (1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,) + tuple(2*i+1 for i in range(k)) + tuple(2*i+2 for i in range(k)))
    return x.reshape(m, 2**k, 2**k) / (2**k)

In [18]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]

In [19]:
n = 256
cb = E8P12_codebook()
W = torch.randn(n, n)
coeffs = hb_transform(W)
coeffs = coeffs.reshape(-1, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hatW = hb_transform(hat_coeffs)
hatW = hatW.reshape(n, n)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 1.0087654699607393


In [20]:
def mats_tensor_product(n):
    rep = int(torch.log2(torch.tensor(n // 8)))
    A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
    B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
    mats = []
    for i in product(range(4), repeat=rep):
        for j in range(64):
            cands = [A2[k] for k in i] + [B8[j]]
            M = cands[0]
            for mat in cands[1:]:
                M = torch.kron(M, mat)
            mats.append(M)
    return torch.stack(mats) / mats[0].norm()

In [21]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 64
cb = E8P12_codebook()
# mats = []
# for i in range(4):
#     for j in range(4):
#         for k in range(4):
#             for l in range(64):
#                 mats.append(torch.kron(torch.kron(torch.kron(A2[i], A2[j]), A2[k]), B8[l]))
# mats = torch.stack(mats) / mats[0].norm() # (4096, 64, 64)
mats = mats_tensor_product(n)
W = torch.randn(n, n)
coeffs = (mats * W).sum(dim=(-2, -1)).reshape(64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(4096)
hatW = (mats * hat_coeffs.view(4096, 1, 1)).sum(dim=0)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error:.4f}")

Error: 0.3037


In [22]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 128
cb = E8P12_codebook()
mats = mats_tensor_product(n)
W = torch.randn(n, n)
coeffs = (mats * W).sum(dim=(-2, -1)).reshape((n * n) // 64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in cliques:
    target = coeffs[:, clique]
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, clique] = hat_clique
hat_coeffs = hat_coeffs.reshape(n * n)
hatW = (mats * hat_coeffs.view(n * n, 1, 1)).sum(dim=0)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error:.4f}")

Error: 0.3033


In [ ]:
A2 = hb_transform(torch.eye(4)) # (4, 2, 2)
B8 = hb_transform(torch.eye(64)) # (64, 8, 8)
n = 64
cb = E8P12_codebook()
mats = mats_tensor_product(n)
W = torch.randn(n, n)
A = torch.randn(n * n, n * n)
H = A.T @ A + n * n * torch.eye(n * n)
mats_flat = mats.reshape(4096, 4096)
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
tr_H_sqrt = torch.diagonal(H_sqrt).sum()
H_sqrt_coeffs = mats_flat @ H_sqrt @ mats_flat.T # (4096, 4096)
mats_flat = mats.reshape(4096, 4096)
flat_coeffs = mats_flat @ W.reshape(4096)
coeffs = flat_coeffs.reshape(64, 64) # (64, 64)
hat_coeffs = torch.zeros_like(coeffs)
for clique in range(8):
    target = coeffs[:, cliques[clique]] # (64, 8)
    curr_clique_inds_flat = [i * 64 + j for j in cliques[clique] for i in range(64)]
    if clique != 0:
        correction = torch.zeros(64, 8)
        for i in range(clique):
            prev_clique_inds_flat = [a * 64 + b for b in cliques[i] for a in range(64)]
            prev_errors = (coeffs[:, cliques[i]] - hat_coeffs[:, cliques[i]]) # (64, 8)
            prev_errors_flat = prev_errors.reshape(-1) # (512,)
            H_sqrt_coeffs_part = H_sqrt_coeffs[curr_clique_inds_flat, :][:, prev_clique_inds_flat] # (512, 512)
            correction_flat = (H_sqrt_coeffs_part @ prev_errors_flat) / tr_H_sqrt # (512,)
            correction += correction_flat.reshape(64, 8)
        target += correction
    hat_clique, _ = cb.quantize(target)
    hat_coeffs[:, cliques[clique]] = hat_clique
hatW_flat = mats_flat.T @ hat_coeffs.reshape(4096)
hatW = hatW_flat.reshape(64, 64) # (64, 64)
error = (W - hatW).norm() / W.norm()
print(f"Error: {error}")

Error: 0.30523870492351285
